# Thick-Walled Cylinder — pyGmshViewer Integration

This notebook extends the `example_plate_pyGmsh` workflow by adding:

1. **VTK Export** — write mesh + results to `.vtu` for any VTK-compatible viewer
2. **pyGmshViewer** launch — standalone Qt/VTK viewer with contours, deformed shapes, and probes
3. **Programmatic probing** — sample field values at points, along lines, and on cut planes

The analysis itself (Lamé problem, quarter annulus) is identical to `example_plate_pyGmsh`.

In [ ]:
from pyGmsh import pyGmsh
from pyGmsh.VTKExport import VTKExport, write_vtu, VTK_TRIANGLE
import numpy as np
import openseespy.opensees as ops

## 1 — Geometry, Mesh & Solve (same as example_plate_pyGmsh)

Condensed into a single cell for brevity. See `example_plate_pyGmsh.ipynb` for detailed commentary.

In [ ]:
# ============================================================
# Parameters
# ============================================================
inner_radius = 100.0    # [mm]
outer_radius = 200.0    # [mm]
lc           = 10.0     # [mm] mesh size
E   = 210.0e3           # [MPa]
nu  = 0.3
p   = 100.0             # [MPa] internal pressure
thk = 1.0               # [mm]

# ============================================================
# pyGmsh: Geometry + Mesh
# ============================================================
g = pyGmsh(model_name="Plate2D", verbose=False)
g.initialize()

pc = g.model.add_point(0, 0, 0, lc=lc)
p1 = g.model.add_point(inner_radius, 0, 0, lc=lc)
p2 = g.model.add_point(outer_radius, 0, 0, lc=lc)
p3 = g.model.add_point(0, outer_radius, 0, lc=lc)
p4 = g.model.add_point(0, inner_radius, 0, lc=lc)

l1 = g.model.add_line(p1, p2)
l2 = g.model.add_arc(p2, pc, p3)
l3 = g.model.add_line(p3, p4)
l4 = g.model.add_arc(p4, pc, p1)

loop = g.model.add_curve_loop([l1, l2, l3, l4])
surf = g.model.add_plane_surface(loop)

pg_symY = g.physical.add(1, [l1], name="Sym_Y")
pg_symX = g.physical.add(1, [l3], name="Sym_X")
pg_pres = g.physical.add(1, [l4], name="Pressure")
pg_plat = g.physical.add(2, [surf], name="Plate")

g.mesh.set_order(1)
g.mesh.generate(2)

# Extract mesh data
fem = g.mesh.get_fem_data(dim=2)
node_tags    = fem['node_tags']
node_coords  = fem['node_coords']
tag_to_idx   = fem['tag_to_idx']
connectivity = fem['connectivity']
elem_tags    = fem['elem_tags']
used_tags    = fem['used_tags']

mesh = g.mesh.get_numbered_mesh(dim=2, method="simple")

bottom_nodes_gmsh = g.physical.get_nodes(1, pg_symY)['tags']
left_nodes_gmsh   = g.physical.get_nodes(1, pg_symX)['tags']
inner_nodes_gmsh  = g.physical.get_nodes(1, pg_pres)['tags']

inner_elems = g.mesh.get_elements(dim=1, tag=l4)
inner_edges = []
for etype, enodes in zip(inner_elems['types'], inner_elems['node_tags']):
    props = g.mesh.get_element_properties(etype)
    inner_edges = enodes.reshape(-1, props['n_nodes']).astype(int)

print(f"Mesh: {mesh.n_nodes} nodes, {mesh.n_elems} elements")

In [ ]:
# ============================================================
# OpenSees: Build model, apply loads, solve
# ============================================================
ops.wipe()
ops.model("basic", "-ndm", 2, "-ndf", 2)

for i in range(mesh.n_nodes):
    nid = int(mesh.node_ids[i])
    ops.node(nid, float(mesh.node_coords[i, 0]), float(mesh.node_coords[i, 1]))

ops.nDMaterial("ElasticIsotropic", 1, E, nu)

for i in range(mesh.n_elems):
    eid = int(mesh.elem_ids[i])
    n1, n2, n3 = [int(n) for n in mesh.connectivity[i]]
    ops.element("tri31", eid, n1, n2, n3, thk, "PlaneStrain", 1)

for gtag in bottom_nodes_gmsh.astype(int):
    sid = mesh.gmsh_to_solver_node.get(int(gtag))
    if sid: ops.fix(sid, 0, 1)

for gtag in left_nodes_gmsh.astype(int):
    sid = mesh.gmsh_to_solver_node.get(int(gtag))
    if sid: ops.fix(sid, 1, 0)

# Consistent pressure loads on inner arc
ops.timeSeries("Linear", 1)
ops.pattern("Plain", 1, 1)

nodal_forces = {}
for edge in inner_edges:
    n1g, n2g = int(edge[0]), int(edge[1])
    i1, i2 = tag_to_idx[n1g], tag_to_idx[n2g]
    x1, y1 = node_coords[i1, 0], node_coords[i1, 1]
    x2, y2 = node_coords[i2, 0], node_coords[i2, 1]
    Le = np.sqrt((x2-x1)**2 + (y2-y1)**2)
    r1 = np.sqrt(x1**2 + y1**2)
    r2 = np.sqrt(x2**2 + y2**2)
    tx1, ty1 = p*x1/r1, p*y1/r1
    tx2, ty2 = p*x2/r2, p*y2/r2
    Fx1 = (Le/6)*(2*tx1+tx2); Fy1 = (Le/6)*(2*ty1+ty2)
    Fx2 = (Le/6)*(tx1+2*tx2); Fy2 = (Le/6)*(ty1+2*ty2)
    o1, o2 = mesh.gmsh_to_solver_node[n1g], mesh.gmsh_to_solver_node[n2g]
    nodal_forces.setdefault(o1, [0., 0.])
    nodal_forces.setdefault(o2, [0., 0.])
    nodal_forces[o1][0] += Fx1; nodal_forces[o1][1] += Fy1
    nodal_forces[o2][0] += Fx2; nodal_forces[o2][1] += Fy2

for nid, (fx, fy) in nodal_forces.items():
    ops.load(nid, fx, fy)

# Solve
ops.constraints("Transformation")
ops.numberer("RCM")
ops.system("BandGeneral")
ops.test("NormDispIncr", 1e-8, 10)
ops.algorithm("Newton")
ops.integrator("LoadControl", 1.0)
ops.analysis("Static")
ok = ops.analyze(1)
print("CONVERGED" if ok == 0 else "FAILED")

In [ ]:
# ============================================================
# Extract results into numpy arrays
# ============================================================
nElem = connectivity.shape[0]
nNode = len(node_tags)

# Displacements
disp = np.zeros((nNode, 2))
for solver_id, gmsh_tag in mesh.solver_to_gmsh_node.items():
    idx = tag_to_idx[gmsh_tag]
    disp[idx, 0] = ops.nodeDisp(solver_id, 1)
    disp[idx, 1] = ops.nodeDisp(solver_id, 2)

# Radial displacement
r_nodes = np.sqrt(node_coords[:, 0]**2 + node_coords[:, 1]**2)
r_safe  = np.where(r_nodes > 1e-12, r_nodes, 1.0)
ur = (node_coords[:, 0]*disp[:, 0] + node_coords[:, 1]*disp[:, 1]) / r_safe

# Element stresses
sig_xx = np.zeros(nElem)
sig_yy = np.zeros(nElem)
sig_xy = np.zeros(nElem)
for solver_eid in range(1, mesh.n_elems + 1):
    gmsh_etag = mesh.solver_to_gmsh_elem[solver_eid]
    idx = elem_tags.index(gmsh_etag)
    s = ops.eleResponse(solver_eid, "stresses")
    sig_xx[idx], sig_yy[idx], sig_xy[idx] = s[0], s[1], s[2]

# Nodal-averaged stress
conn_idx = np.array([[tag_to_idx[n] for n in row] for row in connectivity])
sig_xx_nodal = np.zeros(nNode)
node_count   = np.zeros(nNode)
for e in range(nElem):
    for ln in range(3):
        nidx = conn_idx[e, ln]
        sig_xx_nodal[nidx] += sig_xx[e]
        node_count[nidx]   += 1.0
node_count[node_count == 0] = 1.0
sig_xx_nodal /= node_count

# Von Mises
von_mises_elem = np.sqrt(sig_xx**2 - sig_xx*sig_yy + sig_yy**2 + 3*sig_xy**2)

print(f"u_r  range: [{ur.min():.6f}, {ur.max():.6f}] mm")
print(f"\u03c3_xx range: [{sig_xx.min():.2f}, {sig_xx.max():.2f}] MPa")
print(f"\u03c3_vm range: [{von_mises_elem.min():.2f}, {von_mises_elem.max():.2f}] MPa")

ops.wipe()
print("\nOpenSees wiped — results stored in numpy arrays.")

## 2 — Export to VTU for the Viewer

pyGmsh's `VTKExport` writes `.vtu` files (VTK UnstructuredGrid XML) that any
VTK-compatible tool can open: ParaView, VisIt, and our **pyGmshViewer**.

Two approaches:

| Approach | When to use |
|----------|-------------|
| `VTKExport(g)` class | While Gmsh is still running (extracts mesh internally) |
| `write_vtu()` function | Standalone — pass your own coords + connectivity arrays |

We use the standalone function here since we already have the arrays extracted.

In [ ]:
from pathlib import Path

output_dir = Path("generated")
output_dir.mkdir(exist_ok=True)

# Build 3D displacement vector (VTK needs 3 components)
disp_3d = np.column_stack([disp, np.zeros(nNode)])

# Write the VTU file with all result fields
vtu_path = write_vtu(
    output_dir / "lame_plate_results.vtu",
    node_coords,                       # (N, 3) node coordinates
    conn_idx,                          # (E, 3) 0-based connectivity
    vtk_cell_type=VTK_TRIANGLE,
    point_data={
        "Displacement":   disp_3d,          # (N, 3) vector
        "Displacement_Y":  disp[:, 1],       # (N,) scalar
        "Radial_Disp":     ur,               # (N,) scalar
        "Stress_XX_nodal": sig_xx_nodal,     # (N,) scalar (smoothed)
    },
    cell_data={
        "Stress_XX":   sig_xx,               # (E,) piecewise constant
        "Stress_YY":   sig_yy,
        "Stress_XY":   sig_xy,
        "VonMises":    von_mises_elem,
    },
)

print(f"VTU written: {vtu_path}")
print(f"  Point fields: Displacement (vec), Displacement_Y, Radial_Disp, Stress_XX_nodal")
print(f"  Cell fields:  Stress_XX, Stress_YY, Stress_XY, VonMises")

In [ ]:
# Finalize Gmsh (no longer needed)
g.finalize()

## 3 — Launch pyGmshViewer

The viewer ships with a one-liner `show()` function that works from
notebooks and scripts alike:

```python
from pyGmshViewer import show

# Non-blocking (recommended in Jupyter — notebook keeps running)
show("results.vtu", blocking=False)

# Blocking (script-style — waits until you close the window)
show("results.vtu")

# Multiple files at once
show("mesh.vtu", "modes.pvd", blocking=False)
```

Or from the command line:
```bash
python -m pyGmshViewer generated/lame_plate_results.vtu
```

### Viewer Controls

| Feature | How |
|---------|-----|
| **Contour plot** | Click any field in the Model Tree |
| **Deformed shape** | Controls panel → check "Show Deformed Shape", set scale |
| **Point probe** | Ctrl+P or Probes panel → Point, then click on mesh |
| **Line probe** | Ctrl+L or Probes panel → Line, then two clicks |
| **Plane slice** | Probes panel → Slice X/Y/Z or Interactive Plane |
| **Camera** | XY, XZ, YZ, Iso toolbar buttons or drag to rotate |
| **Screenshot** | Ctrl+S |

In [ ]:
import sys
from pathlib import Path

# Add pyGmshViewer to path if not pip-installed
viewer_root = Path("..").resolve()
if str(viewer_root) not in sys.path:
    sys.path.insert(0, str(viewer_root))

from pyGmshViewer import show

# Launch the viewer (non-blocking so the notebook keeps running)
show(vtu_path, blocking=False)

## 4 — Programmatic Probing (without the GUI)

You can use the probe engine directly in a script or notebook to sample
field values at specific locations. This is useful for:

- Extracting stress at a specific point (e.g., at the inner wall along x-axis)
- Plotting field variation along a line (e.g., radial stress profile)
- Comparing numerical vs analytical solutions at precise locations

The probes use VTK's interpolation, so they work at **any** location —
not just at mesh nodes.

In [ ]:
import pyvista as pv

# Load the VTU we just created
grid = pv.read(str(vtu_path))
print(f"Loaded: {grid.n_points} points, {grid.n_cells} cells")
print(f"Point fields: {list(grid.point_data.keys())}")
print(f"Cell fields:  {list(grid.cell_data.keys())}")

In [ ]:
# ============================================================
# Point Probe: sample at the inner wall along the x-axis
# ============================================================
# This is where the analytical solution gives maximum hoop stress.

probe_location = np.array([inner_radius, 0.0, 0.0])

# Create a single-point probe and sample the mesh
probe_pt = pv.PolyData(probe_location.reshape(1, 3))
sampled = probe_pt.sample(grid)

print("Point Probe at inner wall (x-axis):")
print(f"  Position: ({probe_location[0]}, {probe_location[1]}, {probe_location[2]})")
for name in sampled.point_data:
    val = sampled.point_data[name][0]
    if isinstance(val, np.ndarray) and val.size > 1:
        print(f"  {name}: {val}  (|v| = {np.linalg.norm(val):.6e})")
    else:
        print(f"  {name}: {float(val):.6e}")

In [ ]:
# ============================================================
# Line Probe: radial profile along x-axis (r = R_i to R_o)
# ============================================================
# Sample the stress field along the radial direction and compare
# with the Lamé analytical solution.

n_samples = 200
line = pv.Line(
    [inner_radius, 0, 0],     # start (inner wall)
    [outer_radius, 0, 0],     # end (outer wall)
    resolution=n_samples - 1,
)
sampled_line = line.sample(grid)

# Extract sampled positions and field values
r_sampled = np.array(sampled_line.points)[:, 0]  # x = r along this line
sig_xx_sampled = np.array(sampled_line.point_data["Stress_XX_nodal"])
ur_sampled = np.array(sampled_line.point_data["Radial_Disp"])

print(f"Line probe: {len(r_sampled)} samples from r={inner_radius} to r={outer_radius}")
print(f"  Stress_XX range: [{sig_xx_sampled.min():.2f}, {sig_xx_sampled.max():.2f}] MPa")

In [ ]:
# ============================================================
# Analytical solution (Lamé)
# ============================================================
# Along the x-axis (theta=0):
#   sigma_rr = sigma_xx = A + B/r^2
#   sigma_tt = sigma_yy = A - B/r^2
#   u_r = (1/E) * [(1+nu)*B/r + (1-2*nu)*(1+nu)*A*r]
#
# where A = p*a^2 / (b^2 - a^2),  B = -p*a^2*b^2 / (b^2 - a^2)

a, b = inner_radius, outer_radius
A_lame = p * a**2 / (b**2 - a**2)
B_lame = -p * a**2 * b**2 / (b**2 - a**2)

r_analytical = np.linspace(inner_radius, outer_radius, 500)
sig_rr_exact = A_lame + B_lame / r_analytical**2
sig_tt_exact = A_lame - B_lame / r_analytical**2
ur_exact = (1/E) * ((1+nu)*B_lame/r_analytical + (1-2*nu)*(1+nu)*A_lame*r_analytical)

print(f"Analytical at inner wall: \u03c3_rr = {sig_rr_exact[0]:.2f}, \u03c3_tt = {sig_tt_exact[0]:.2f} MPa")
print(f"Analytical at outer wall: \u03c3_rr = {sig_rr_exact[-1]:.2f}, \u03c3_tt = {sig_tt_exact[-1]:.2f} MPa")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# (a) Radial stress profile: FEM probe vs analytical
ax = axes[0]
ax.plot(r_analytical, sig_rr_exact, 'k-', lw=2, label=r'$\sigma_{rr}$ (Lamé)')
ax.plot(r_analytical, sig_tt_exact, 'k--', lw=2, label=r'$\sigma_{\theta\theta}$ (Lamé)')
ax.plot(r_sampled, sig_xx_sampled, 'ro', ms=3, alpha=0.5,
        label=r'$\sigma_{xx}$ (FEM probe)')
ax.set_xlabel('r [mm]')
ax.set_ylabel('Stress [MPa]')
ax.set_title('Radial Stress Profile — Line Probe vs Analytical')
ax.legend()
ax.grid(True, alpha=0.3)

# (b) Radial displacement: FEM probe vs analytical
ax = axes[1]
ax.plot(r_analytical, ur_exact, 'k-', lw=2, label=r'$u_r$ (Lamé)')
ax.plot(r_sampled, ur_sampled, 'bo', ms=3, alpha=0.5,
        label=r'$u_r$ (FEM probe)')
ax.set_xlabel('r [mm]')
ax.set_ylabel('Displacement [mm]')
ax.set_title('Radial Displacement — Line Probe vs Analytical')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(str(output_dir / "lame_probe_comparison.png"), dpi=150)
plt.show()
print(f"\nPlot saved: {output_dir / 'lame_probe_comparison.png'}")

## 5 — Plane Probe (Slice)

Cut the mesh with a plane to inspect field values on the cross-section.
Useful for 3D models where you want to see internal stress distributions.

For this 2D problem (z=0 plane), a radial cut line is more natural,
but we demonstrate the API for completeness.

In [ ]:
# Slice the mesh along the 45-degree line (normal = [-1, 1, 0])
# This cuts through the quarter annulus diagonally.

origin = np.array([150.0, 0.0, 0.0])  # midpoint of radial extent
normal = np.array([0.0, 1.0, 0.0])    # cut along y-direction

sliced = grid.slice(normal=normal, origin=origin)

if sliced.n_points > 0:
    print(f"Plane slice: {sliced.n_points} points, {sliced.n_cells} cells")
    print(f"Fields on slice: {list(sliced.point_data.keys())}")
    
    # Extract data along the slice for plotting
    slice_coords = np.array(sliced.points)
    slice_y = slice_coords[:, 1]
    slice_stress = np.array(sliced.point_data.get("Stress_XX_nodal", []))
    
    if len(slice_stress) > 0:
        # Sort by y for clean plotting
        order = np.argsort(slice_y)
        plt.figure(figsize=(8, 4))
        plt.plot(slice_y[order], slice_stress[order], 'g.-', lw=1.5)
        plt.xlabel('y [mm]')
        plt.ylabel(r'$\sigma_{xx}$ [MPa]')
        plt.title(f'Stress along plane slice at x = {origin[0]} mm')
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()
else:
    print("Slice returned no points (plane may not intersect the mesh).")

## 6 — Quick Inline Visualization with PyVista

If you don't need the full viewer GUI, PyVista can render directly
in the notebook (with `jupyter_backend='static'` or `'trame'`).

In [ ]:
# Static inline plot (works in any Jupyter environment)
pv.set_jupyter_backend('static')

plotter = pv.Plotter(off_screen=True, window_size=[900, 500])
plotter.set_background('#1a1a2e', top='#16213e')

plotter.add_mesh(
    grid,
    scalars='VonMises',
    cmap='turbo',
    show_edges=True,
    edge_color='#2C4A6E',
    line_width=1,
    scalar_bar_args={
        'title': 'Von Mises [MPa]',
        'color': 'white',
    },
)
plotter.add_axes(color='white')
plotter.view_xy()

# Save and display
img_path = str(output_dir / 'lame_vonmises.png')
plotter.screenshot(img_path)
plotter.close()

from IPython.display import Image, display
display(Image(filename=img_path, width=800))
print(f"Screenshot saved: {img_path}")

In [ ]:
# Deformed shape with displacement contour
plotter = pv.Plotter(off_screen=True, window_size=[900, 500])
plotter.set_background('#1a1a2e', top='#16213e')

# Undeformed reference (wireframe)
plotter.add_mesh(
    grid, style='wireframe', color='#888888',
    line_width=1, opacity=0.3, label='Undeformed',
)

# Deformed mesh (scale factor = 500 for visibility)
scale = 500.0
deformed = grid.copy()
deformed.points = np.array(grid.points) + scale * np.array(grid.point_data['Displacement'])

plotter.add_mesh(
    deformed,
    scalars='Radial_Disp',
    cmap='coolwarm',
    show_edges=True,
    edge_color='#803D00',
    line_width=1,
    scalar_bar_args={
        'title': f'u_r [mm] (x{int(scale)})',
        'color': 'white',
    },
)
plotter.add_axes(color='white')
plotter.view_xy()

img_path = str(output_dir / 'lame_deformed.png')
plotter.screenshot(img_path)
plotter.close()

display(Image(filename=img_path, width=800))
print(f"Deformed shape saved: {img_path}")

## Summary

The full pipeline from geometry to interactive visualization:

```
pyGmsh (geometry + mesh)
    │
    ├── NumberedMesh (contiguous IDs + maps)
    │       │
    │       └── OpenSees (solve)
    │               │
    │               └── numpy arrays (disp, stress, ...)
    │
    └── VTKExport (write .vtu)
            │
            ├── pyGmshViewer (Qt + VTK GUI with probes)
            ├── PyVista (inline notebook plots)
            └── ParaView (.vtu is native format)
```

**Key files generated:**
- `generated/lame_plate_results.vtu` — mesh + all result fields
- `generated/lame_probe_comparison.png` — FEM vs analytical comparison
- `generated/lame_vonmises.png` — Von Mises contour screenshot
- `generated/lame_deformed.png` — deformed shape screenshot